<a href="https://colab.research.google.com/github/shelxty/basic-asl-translator/blob/main/basicASL_detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Basic Real-Time Sign Language Detection w/ Tensorflow

**NOTE:** If running from a different device, make sure to upgrade your fsspec, pandas, and datasets (huggingface API) + decord and tensorflow via these commands:

- `pip install --upgrade fsspec`
- `pip install --upgrade pandas`
- `pip install --upgrade tensorflow`
- `pip install decord`


Then you can check their current versions / upgraded versions with `pip show library_here` (like `pip show tensorflow`).


Note that after upgrading, you have to restart the kernel to use the upgraded versions.


Additionally, ensure that numpy 1.26.4 is being used.

In [1]:
# importing all the necesary libraries
import numpy as np
import cv2 # this is opencv
import os # helps work w/ file paths
import time # we need to take a break between the images so this spaces time apart yesyesyes
import uuid # name image files
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
import tensorflow as tf
# WHY DO I EKEP GETINTG ERRORS HER AISHFOIMHITCS


#%pip install --upgrade numpy pandas scipy matplotlib scikit-learn
from tensorflow.keras import layers, models
from tensorflow.keras.models import Sequential
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.utils import to_categorical
# import pathlib
from PIL import Image

In [32]:
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

#### Getting the Dataset

I'm using a dataset from HuggingFace: https://huggingface.co/datasets/ZahidYasinMittha/American-Sign-Language-Dataset


It's 108,618 videos representing 2,208 ASL words, each word with a minimum of 30 videos.

In [3]:
pip install decord

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.6/13.6 MB 112.5 MB/s eta 0:00:00


In [4]:
from datasets import load_dataset
import decord
print(f"Decord version: {decord.__version__}")

Decord version: 0.6.0


In [5]:
print(f"Pandas version: {pd.__version__}")

Pandas version: 2.2.2


In [6]:
pip show decord

Name: decord
Version: 0.6.0
Summary: Decord Video Loader
Home-page: https://github.com/dmlc/decord
Author: 
Author-email: 
License: APACHE
Location: /usr/local/lib/python3.12/dist-packages
Requires: numpy
Required-by: 


In [7]:
from huggingface_hub import hf_hub_download

Link: https://huggingface.co/datasets/ZahidYasinMittha/American-Sign-Language-Dataset/resolve/main/Aslense%20Dataset.csv

In [8]:
csv_path = hf_hub_download(
    repo_id = "ZahidYasinMittha/American-Sign-Language-Dataset",
    filename = "Aslense Dataset.csv",
    repo_type = "dataset"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Aslense%20Dataset.csv:   0%|          | 0.00/3.68M [00:00<?, ?B/s]

In [9]:
df = pd.read_csv(csv_path)
df.head(10)

,word,videos
0,a,A.mp4
1,a,a_7.mp4
2,a,a_2.mp4
3,a,a_4.mp4
4,a,a_1.mp4
5,a,a_5.mp4
6,a,a_6.mp4
7,a,a_3.mp4
8,a,A_video_2.mp4
9,a,A_video_0.mp4


In [10]:
print("Total number of videos: ", len(df))
print("Columns: ", df.columns.tolist())

Total number of videos:  108618
Columns:  ['word', 'videos']


In [11]:
df.describe()

,word,videos
count,108618,108618
unique,2207,108617
top,erase,h_5.mp4
freq,160,2


In [12]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 108618 entries, 0 to 108617
Data columns (total 2 columns):
 #   Column  Non-Null Count   Dtype 
---  ------  --------------   ----- 
 0   word    108618 non-null  object
 1   videos  108618 non-null  object
dtypes: object(2)
memory usage: 1.7+ MB


In [13]:
df.isnull().sum() # no null rows phew

,0
word,0
videos,0


Now that we have the dataset, we have to build a streaming DataLoader so that we can access the videos without downloading all 108 thousand videos.

With decord, we can decode each video on the fly.

In [14]:
# sample = next(iter(ds))
# print(sample.keys())

In [15]:
# print(sample.keys())
# print(type(sample["video"]))

In [16]:
import torch
from torch.utils.data import IterableDataset, DataLoader
import io

In [35]:
ds_debug = load_dataset("ZahidYasinMittha/American-Sign-Language-Dataset", streaming = True, split = "train")

for sample in ds_debug:
  print(type(sample["video"]))
  print(dir(sample["video"]))
  print(sample.keys())
  break

Resolving data files:   0%|          | 0/22 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/25 [00:00<?, ?it/s]

<class 'torchcodec.decoders._video_decoder.VideoDecoder'>
['__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__getitem__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__len__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__str__', '__subclasshook__', '__weakref__', '_begin_stream_seconds', '_cpu_fallback', '_decoder', '_end_stream_seconds', '_get_key_frame_indices', '_getitem_int', '_getitem_slice', '_hf_encoded', '_num_frames', 'cpu_fallback', 'get_all_frames', 'get_frame_at', 'get_frame_played_at', 'get_frames_at', 'get_frames_in_range', 'get_frames_played_at', 'get_frames_played_in_range', 'metadata', 'stream_index']
dict_keys(['video'])


In [17]:
class ASLStreamingDataset(IterableDataset):
  def __init__(self, word_to_idx, split = "train", buffer_size = 200, num_frames = 16, frame_size = (112, 112)):
    super().__init__()
    self.word_to_idx = word_to_idx
    self.split = split
    self.buffer_size = buffer_size
    self.num_frames = num_frames
    self.frame_size = frame_size

  def __iter__(self):
    ds = load_dataset("ZahidYasinMittha/American-Sign-Language-Dataset", streaming = True, split = self.split)

    ds = ds.shuffle(seed = 42, buffer_size = self.buffer_size)

    for sample in ds: # iterate thru the whoel datset
      video_bytes = sample["video"]["bytes"]
      label_word = sample["label"]

      if label_word not in self.word_to_idx:
        continue

      label_idx = self.word_to_idx[label_word]

      tensor = self._preprocess(video_bytes)

      if tensor is None:
        continue # skip corrupted vids

      yield tensor, torch.tensor(label_idx, dtype = torch.long)

    def _preprocess(self, video_bytes): # video_bytes would be all the raw bytes of a .mp4 file
    # this function will return the float tensor of shape (3, T, H, W), values in [0, 1], or None if the video can't be read
      import tempfile
      with tempfile.NamedTemporaryFile(suffix = ".mp4", delete = False) as f: # opencv can't read from the raw bytes directly, so we wrap them up in a numpy array. just write it all to a temp file for now
        f.write(video_bytes)
        tmp_path = f.name

      try:
        cap = cv2.VideoCapture(tmp_path)
        if not cap.isOpened():
          return None # should the vid be corrupted, return nothing

        all_frames = [] # this is all the frames in the video
        while True:
          ret, frame = cap.read()
          # ret = true if a fram is successfuly read
          # ret = false at  the end of the video
          if not ret:
            break

          frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB) # since opencv reads in bgr order, convert it to rgb
          all_frames.append(frame)

        cap.release()

        if len(all_frames) == 0:
          return None

          # need to sample all the farmes evenly
          # like if we have 48 frames and want 16, we'll pick the indices [0, 3, 6, 9, ... 42, 45]
          # & if we have 8 frames but want 16, we'll repeat some frames

          total = len(all_frames)
          indices = np.linspace(0, total - 1, self.num_frames, dtype = int) # provides num_frames evenly spaced numbers between 0 and total - 1
          sampled = [all_frames[i] for i in indices]

          # resize each frame
          H, W = self.frame_size
          resized = [cv2.resize(f, (W, H)) for f in sampled]

          video_array = np.stack(resized, axis = 0).astype(np.float32)

          video_array /= 255 # normalize from [0, 255] to [0, 1]

          torch = torch.from_numpy(video_array)
          tensor = tensor.permute(3, 0, 1, 2)

          return tensor

      finally:
        os.unlink(tmp_path)

In [18]:
unique_words = sorted(df["word"].unique())
word_to_idx = {word: i for i, word in enumerate(unique_words)}
idx_to_word = {i: word for word, i in word_to_idx.items()}

num_classes = len(word_to_idx)
print(f"Number of classes: {num_classes}")

Number of classes: 2207


In [19]:
train_dataset = ASLStreamingDataset(word_to_idx, split = "train", buffer_size = 200)
test_dataset = ASLStreamingDataset(word_to_idx, split = "test", buffer_size = 200)

In [20]:
train_loader = DataLoader(
    train_dataset,
    batch_size = 8, # process 8 vids at a time
    num_workers = 2,
    pin_memory = True # cpu to gpu transfer
)

test_loader = DataLoader(
    test_dataset,
    batch_size = 8, # process 8 vids at a time
    num_workers = 2,
    pin_memory = True # cpu to gpu transfer
)

### Model

2D cnn learns spatial patterns (e.g. edges, shapes, textures)
* 2D conv kernel is a 3x3 patch

3D CNN learns temporal patterns (how things move over time)
* 3D conv kernel is a 3x3x3 patch

R3D_18 = ResNet-18; it's adapted for video.

https://github.com/kryptologyst/Action-Recognition-in-Videos already trained it on 400 classes for action; we'll take his architecture and use it here.

In [21]:
import torch
import torch.nn as nn
from torchvision.models.video import r3d_18, R3D_18_Weights

In [22]:
class ASLClassifier(nn.Module):
  def __init__(self, num_classes, freeze_backbone = True):
    # we use the numbe of classes (num_classes) for the number of asl words (in this case, 2208)
    # the architecture of f3d_18 has a last layer that's trained on 400 kinetic's classes
    # but we'll change that last layer to train on 2208 asl words

    super().__init__()
    self.backbone = r3d_18(weights = R3D_18_Weights.DEFAULT) # r3d 18 weights default gives best available pretrained weights & downloads about 45 megabtes of websites from pytorch models

    if freeze_backbone:
      for param in self.backbone.parameters():
        param.requires_grad = False

    in_features = self.backbone.fc.in_features
    # original fc layer is backbone.fc = Linear(512, 400) but we need to replace the 400 with the number of clsses we have (2208)


    self.backbone.fc = nn.Sequential(
        nn.Dropout(p = 0.3),
        nn.Linear(in_features, num_classes)
    )

  def forward(self, x):
    return self.backbone(x)

  def unfreeze_backbone(self):
    for param in self.backbone.parameters():
      param.requires_grad = True
    print("Backbone unfrozen. All params are now trainable")

In [23]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

model = ASLClassifier(num_classes=num_classes, freeze_backbone=True)
model = model.to(device)

Using device: cuda
Downloading: "https://download.pytorch.org/models/r3d_18-b3b3357e.pth" to /root/.cache/torch/hub/checkpoints/r3d_18-b3b3357e.pth


100%|██████████| 127M/127M [00:00<00:00, 165MB/s]


In [24]:
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params: ,}")

Total parameters: 34,298,463
Trainable parameters:  1,132,191


In [25]:
dummy_input = torch.randn(2, 3, 16, 112, 112).to(device)
with torch.no_grad():
  dummy_output = model(dummy_input)
print(f"Output shape: {dummy_output.shape}")

Output shape: torch.Size([2, 2207])


Training loop:
- CrossEntropyLoss loss function
- Adam optimizer
- Gradient scaler (GradScaler)

In [26]:
from torch.cuda.amp import GradScaler, autocast

In [27]:
def train_model(model, train_loader, test_loader, num_classes, num_epochs = 20, lr = 1e-3, unfreeze_epoch = 5, checkpoint_dir = "checkpoints"):
  os.makedirs(checkpoint_dir, exist_ok=True)
  device = next(model.parameters()).device # mkdir on whatever device model's on

  criterion = nn.CrossEntropyLoss(label_smoothing = 0.1)

  optimizer = torch.optim.AdamW(
      filter(lambda p: p.requires_grad, model.parameters()),
      lr = lr,
      weight_decay = 1e-4
  )

  scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
      optimizer,
      T_max = num_epochs,
      eta_min = 1e-6
  )

  scaler = GradScaler(enabled = torch.cuda.is_available())

  best_val_acc = 0.0

  for epoch in range(1, num_epochs + 1):
    epoch_start = time.time()

    if epoch == unfreeze_epoch:
      model.unfreeze_backbone()
      optimizer = torch.optim.AdamW(
          model.parameters(),
          lr = 1e-5,
          weight_decay = 1e-4
      )
      scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
          optimizer,
          T_max = (num_epochs - unfreeze_epoch),
          eta_min = 1e-7
      )
      print(f"\nEpoch {epoch}: Switched to full fine-tuning\n")

      # - -- - - - - - - -
      model.train()
      # turns on dropout and batch normalization training behavior
      train_loss = 0.0
      train_correct = 0
      train_total = 0

      for step, (videos, labels) in enumerate(train_loader):
        videos = videos.to(device, non_blocking = True)
        labels = labels.to(device, non_blocking = True)

        optimizer.zero_grad()

        with autocast(enabled = torch.cuda.is_available()):
          logits = model(videos)
          loss = criterion(logits, labels)

        scaler.scale(loss).backward()

        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm = 1.0)

        scaler.step(optimizer)
        scaler.update()

        train_loss += loss.item()
        predicted = logits.argmax(dim = 1)
        train_correct += (predicted == labels).sum().item()
        train_total += labels.size(0)

        if (step + 1) % 50 == 0:
          running_acc = train_correct / train_total * 100
          print(
              f"Epoch {epoch}, step {step + 1} \n"
              f"Loss: {loss.item():.4f} \n"
              f"Accuracy: {running_acc:.2f}%"
          )

      avg_train_loss = train_loss / (step + 1)
      avg_train_acc = train_correct / train_total * 100

      val_loss, val_top1, val_top5 = evaluate(model, test_loader, criterion, device)

      scheduler.step()

      elapsed = time.time() - epoch_start
      current_lr = optimizer.param_groups[0]['lr']

      print(
          f"\n\nEpoch {epoch}/{num_epochs} ({elapsed:.0f}s) \n"
          f"LR: {current_lr:.2e} \n"
          f"Train loss: {avg_train_loss:.4f} | Training accuracy: {avg_train_acc:.2f}% \n"
          f"Validation loss: {val_loss:.4f} | Validation top-1: {val_top1:.2f}%, validation top-5: {val_top5:.2f}%"
      )

      if val_top1 > best_val_acc:
        best_val_acc = val_top1
        checkpoint_path = os.path.join(checkpoint_dir, "best_model.pt")
        torch.save({
            "epoch": epoch,
            "model_state": model.state_dict(),
            "optimizer": optimizer.state_dict(),
            "val_top1": val_top1,
            "word_to_idx": word_to_idx,
        }, checkpoint_path)
        print(f"Saved new best model: validation top-1 {val_top1:.2f}% --> {checkpoint_path}\n")

  print(f"Training complete. Best validation top-1: {best_val_acc:.2f}%")



In [33]:
def evaluate(model, loader, criterion, device):
  model.eval()
  total_loss = 0.0
  top1_correct = 0
  top5_correct = 0
  total = 0

  with torch.no_grad():
    for videos, labels in loader:
      videos = videos.to(device, non_blocking = True)
      labels = labels.to(device, non_blocking = True)

      with autocast(enabled = torch.cuda.is_available()):
        logits = model(videos)
        loss = criterion(logits, labels)

      total_loss += loss.item()

      # top 1 accuracy
      pred_top1 = logits.argmax(dim = 1)
      top1_correct += (pred_top1 == labels).sum().item()

      # top 5 accuracy
      pred_top5 = logits.topk(5, dim = 1).indices
      for i in range(labels.size(0)):
        if labels[i].item() in pred_top5[i].tolist():
          top5_correct += 1

      total += labels.size(0)

    avg_loss = total_loss / len(loader)
    top1_acc = top1_correct / total * 100
    top5_acc = top5_correct / total * 100

    return avg_loss, top1_acc, top5_acc


In [34]:
train_model(
    model = model,
    train_loader = train_loader,
    test_loader = test_loader,
    num_classes = num_classes,
    num_epochs = 20,
    lr = 1e-3,
    unfreeze_epoch = 5,
    checkpoint_dir = "checkpoints"
)

Backbone unfrozen. All params are now trainable

Epoch 5: Switched to full fine-tuning



/tmp/ipykernel_5982/2970731622.py:19: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler(enabled = torch.cuda.is_available())
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7eb4e29a8180>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
    if w.is_alive():
       ^^^^^^^^^^^^
Exception ignored in:   File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
<function _MultiProcessingDataLoaderIter.__del__ at 0x7eb4e29a8180>
    Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
assert self._parent_pid == os.getpid(), 'can only test

Resolving data files:   0%|          | 0/22 [00:02<?, ?it/s]

Resolving data files:   0%|          | 0/22 [00:03<?, ?it/s]

Resolving data files:   0%|          | 0/25 [00:00<?, ?it/s]

Too many dataloader workers: 2 (max is dataset.num_shards=1). Stopping 1 dataloader workers.


Resolving data files:   0%|          | 0/25 [00:00<?, ?it/s]

TypeError: Caught TypeError in DataLoader worker process 0.
Original Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/_utils/worker.py", line 358, in _worker_loop
    data = fetcher.fetch(index)  # type: ignore[possibly-undefined]
           ^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/_utils/fetch.py", line 35, in fetch
    data.append(next(self.dataset_iter))
                ^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_5982/2235359810.py", line 16, in __iter__
    video_bytes = sample["video"]["bytes"]
                  ~~~~~~~~~~~~~~~^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torchcodec/decoders/_video_decoder.py", line 315, in __getitem__
    raise TypeError(
TypeError: Unsupported key type: <class 'str'>. Supported types are int and slice.


---------------